# Week 7 · Day 2 — Loading data at scale (`stages` + `COPY INTO`)

*You won't type a 200,000-row export by hand. You bulk-load a file — Snowflake's stage + `COPY INTO`.*

**By the end you'll have shipped:** a table **loaded from a CSV file** in one shot — the real Snowflake load path (`PUT` → `@stage` → `COPY INTO`) explained, and its local equivalent run end-to-end, with a row-count check.

### 📋 Lesson card

| | |
|---|---|
| **Module** | M4 · Data & Snowflake (Week 7) |
| **Prerequisites** | W7D1 (`CREATE TABLE`, types) |
| **Est. time** | ~30 min |
| **Capstone slice** | **Ingest** — getting a firm's matter export *into* the warehouse |
| **Difficulty** | Core + `Go Deeper 🔧` |
| **Runs offline?** | ✅ Yes — the Snowflake load path is shown as real SQL; a local load runs live |

### 🎯 Learning objectives

By the end you'll be able to:
- Describe Snowflake's bulk-load path: **`PUT` a file → a `stage` → `COPY INTO` a table**.
- Explain what a **stage** and a **file format** are.
- Load a **CSV into a table** and verify the row count.
- Handle the basics: header rows, delimiters, and **`ON_ERROR`** behavior.
- Know when to bulk-load vs. `INSERT` (and the local twin, `read_csv`).

### ⚖️ Why it matters

The firm's matters won't arrive as SQL — they'll arrive as a **CSV or a stack of files** exported from a document system. "Getting the data in" (ingestion) is step one of *Matter Intelligence*, and doing it by hand-typed `INSERT`s is hopeless at scale. Snowflake's `COPY INTO` loads millions of rows from files fast; today you learn the shape of that path and run the local equivalent.

### ⚙️ Setup

Loads the usual `run_sql` helper and a `coffee_orders` frame. We'll then **write that frame out to a CSV** and load it back from the file — simulating a firm's export landing on disk.

> 🔒 *Synthetic data only — this coffee set (and the matters set) is fake. Never load real client or privileged data into a teaching notebook.*

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import pandas as pd

# Load .env if python-dotenv is present (optional — the notebook runs fine without it).
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# --- Pick the backend: Snowflake if credentials exist in .env, else local DuckDB ---
# You write the SAME SQL either way; the backend is invisible.
SNOWFLAKE_READY = all(os.environ.get(k) for k in ("SNOWFLAKE_ACCOUNT", "SNOWFLAKE_USER", "SNOWFLAKE_PASSWORD"))
BACKEND = "snowflake" if SNOWFLAKE_READY else "duckdb"

def _find(fname):
    for base in ("../../data/", "data/", ""):
        if os.path.exists(base + fname):
            return base + fname
    return None

# Tables this lesson needs — loaded from Training/data/, with a tiny built-in fallback.
FALLBACK = {
    'coffee_orders': [{'order_id': 'O-5001', 'date': '2026-03-07', 'item': 'Cappuccino', 'size': 'S', 'category': 'Espresso Drink', 'price': 3.75, 'payment': 'Cash', 'store': 'Downtown'}, {'order_id': 'O-5002', 'date': '2026-03-07', 'item': 'Mocha', 'size': 'S', 'category': 'Espresso Drink', 'price': 4.5, 'payment': 'Card', 'store': 'Airport'}, {'order_id': 'O-5003', 'date': '2026-03-05', 'item': 'Cappuccino', 'size': 'L', 'category': 'Espresso Drink', 'price': 5.25, 'payment': 'Card', 'store': 'Downtown'}, {'order_id': 'O-5004', 'date': '2026-03-05', 'item': 'Croissant', 'size': 'M', 'category': 'Food', 'price': 3.25, 'payment': 'App', 'store': 'Uptown'}, {'order_id': 'O-5005', 'date': '2026-03-06', 'item': 'Latte', 'size': 'L', 'category': 'Espresso Drink', 'price': 5.5, 'payment': 'App', 'store': 'Downtown'}],
}
frames = {}
for _name, _rows in FALLBACK.items():
    _p = _find(_name + ".csv")
    frames[_name] = pd.read_csv(_p) if _p else pd.DataFrame(_rows)

if BACKEND == "duckdb":
    try:
        import duckdb
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
        import duckdb
    _con = duckdb.connect(":memory:")                 # a private, in-memory warehouse
    for _name, _df in frames.items():
        _con.register("_src_" + _name, _df)
        _con.execute(f"CREATE OR REPLACE TABLE {_name} AS SELECT * FROM _src_{_name}")
    def run_sql(sql: str) -> pd.DataFrame:
        """Run SQL and return the result as a pandas DataFrame."""
        return _con.execute(sql).df()
else:
    import snowflake.connector
    from snowflake.connector.pandas_tools import write_pandas
    _con = snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"], user=os.environ["SNOWFLAKE_USER"],
        password=os.environ["SNOWFLAKE_PASSWORD"], warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE"),
        database=os.environ.get("SNOWFLAKE_DATABASE"), schema=os.environ.get("SNOWFLAKE_SCHEMA"))
    for _name, _df in frames.items():
        write_pandas(_con, _df, _name.upper(), auto_create_table=True, overwrite=True, quote_identifiers=False)
    def run_sql(sql: str) -> pd.DataFrame:
        """Run SQL and return the result as a pandas DataFrame."""
        cur = _con.cursor(); cur.execute(sql); return cur.fetch_pandas_all()

print(f"✅ Ready. Backend = {BACKEND.upper()} · tables: {', '.join(frames)}")

In [ ]:
# Simulate an exported file arriving on disk (this stands in for the firm's CSV export).
EXPORT_PATH = "_orders_export.csv"
frames["coffee_orders"].to_csv(EXPORT_PATH, index=False)
print(f"📄 wrote {EXPORT_PATH} ({len(frames['coffee_orders'])} rows) — pretend this came from a document system")

### 1 · The Snowflake load path — `PUT` → stage → `COPY INTO`

On a real account, bulk loading has three moves:

1. **`PUT`** — upload your local file into a **stage** (Snowflake's staging area for files). *Run from SnowSQL/the connector, not plain SQL.*
2. **File format** — tell Snowflake how to read the file (CSV? header row? comma-delimited?).
3. **`COPY INTO`** — parse the staged file and load its rows into a table.

The SQL looks like this (this is what you'd run on Snowflake):

```sql
-- 1. a place to stage files, and a reusable CSV format
CREATE OR REPLACE STAGE matter_stage;
CREATE OR REPLACE FILE FORMAT csv_fmt
    TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1;

-- 2. upload the local file into the stage (connector/SnowSQL command)
PUT file:///path/coffee_orders.csv @matter_stage;

-- 3. load staged file(s) into the table
COPY INTO coffee_orders
    FROM @matter_stage
    FILE_FORMAT = (FORMAT_NAME = csv_fmt)
    ON_ERROR = 'CONTINUE';
```

**`SKIP_HEADER = 1`** ignores the column-name row; **`ON_ERROR = 'CONTINUE'`** skips bad rows instead of failing the whole load. We can't `PUT` a file from this offline notebook, so next we run the **local equivalent** — same idea, one command.

### 2 · Load a CSV into a table (runs here, both backends)

`load_csv(table, path)` below does what `COPY INTO` does — reads a CSV file and fills a table — using the right mechanism for whichever backend you're on. Offline (DuckDB) it uses `read_csv_auto`; on Snowflake it stages the rows for you.

In [ ]:
def load_csv(table: str, path: str):
    """Load a CSV file into `table`. The local twin of Snowflake's COPY INTO."""
    if BACKEND == "duckdb":
        # DuckDB reads the file directly — the offline COPY INTO.
        _con.execute(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM read_csv_auto('{path}')")
    else:
        # Snowflake: stage the rows and bulk-load (auto-creates the table).
        from snowflake.connector.pandas_tools import write_pandas
        write_pandas(_con, pd.read_csv(path), table.upper(),
                     auto_create_table=True, overwrite=True, quote_identifiers=False)
    return run_sql(f"SELECT COUNT(*) AS rows_loaded FROM {table}")

# Load the "export" into a brand-new table.
load_csv("orders_loaded", EXPORT_PATH)

**What just happened:** one call parsed the CSV and populated `orders_loaded`. The returned count is your **load receipt** — always verify it against the file. Now it's a normal table:

In [ ]:
run_sql("SELECT order_id, item, price, store FROM orders_loaded ORDER BY price DESC LIMIT 5")

### 3 · Always reconcile — did every row land?

Loading silently drops rows more often than you'd think (bad encodings, extra columns, type mismatches). The habit that saves you: **compare the file's row count to the table's.**

In [ ]:
file_rows  = len(pd.read_csv(EXPORT_PATH))
table_rows = run_sql("SELECT COUNT(*) AS n FROM orders_loaded")["n"][0]
print(f"file: {file_rows} rows   |   table: {table_rows} rows   |   match: {file_rows == table_rows}")

> **`Go Deeper 🔧` — appending vs. replacing, and messy files.**
> - Our `load_csv` **replaces** the table each run (`CREATE OR REPLACE`). To **append** new rows instead, you'd `COPY INTO` an existing table (Snowflake tracks which files it already loaded, so re-running won't double-load).
> - Real exports are messy — Week 2's cleaning lesson applies. Load raw into a **staging** table, clean with SQL/pandas, then write the tidy result to the final table (a mini "ELT").
> - **`ON_ERROR`** options: `'ABORT_STATEMENT'` (fail the load), `'CONTINUE'` (skip bad rows), `'SKIP_FILE'` (skip the whole file).

> **`Common pitfalls ⚠️`**
>
> - **Forgetting `SKIP_HEADER = 1`** loads the header as a data row — check row counts.
> - **Column order / count must match** the table (or map them explicitly). A wrong count is a classic silent failure.
> - **Dates & numbers** need a format the loader understands (`2026-01-12`, `18500.00`), or they arrive as text.
> - **Bulk-load files; don't `INSERT` row-by-row** for anything large — `COPY INTO` is dramatically faster.

### ✍️ Your turn

In [ ]:
# TODO 1: write frames["coffee_orders"] to a file called "_menu_like.csv" (any subset is fine)
#         (or reuse EXPORT_PATH)

# TODO 2: load it into a new table `orders_v2` with load_csv(...)

# TODO 3: reconcile — assert the table row count equals the file row count


<details><summary>✅ Show solution</summary>

```python
# 1
path = "_orders_v2.csv"
frames["coffee_orders"].to_csv(path, index=False)

# 2
load_csv("orders_v2", path)

# 3
file_rows  = len(pd.read_csv(path))
table_rows = run_sql("SELECT COUNT(*) AS n FROM orders_v2")["n"][0]
assert file_rows == table_rows, f"mismatch: {file_rows} vs {table_rows}"
print("✅ reconciled:", table_rows, "rows")
```
</details>

### 🚀 Build the artifact — a reusable, reconciled loader

Wrap load + verify into one function you can point at any export. This is the ingestion step of *Matter Intelligence*: hand it a file, get a loaded table and a receipt.

In [ ]:
def ingest(table: str, path: str) -> dict:
    """Load a CSV into `table` and return a reconciliation receipt."""
    load_csv(table, path)
    file_rows  = len(pd.read_csv(path))
    table_rows = int(run_sql(f"SELECT COUNT(*) AS n FROM {table}")["n"][0])
    receipt = {"table": table, "file_rows": file_rows,
               "table_rows": table_rows, "ok": file_rows == table_rows}
    print(f"📥 loaded {table_rows} rows into {table} — reconciled: {receipt['ok']}")
    return receipt

ingest("matters_loaded", EXPORT_PATH)   # (using the coffee export as a stand-in file)

> **🔗 Your world.** Swap the file: point `ingest("matters", "matters_export.csv")` at a real matter export and you've loaded the firm's matters into Snowflake — the front door of *Matter Intelligence*. On a live account the same job is `COPY INTO matters FROM @matter_stage` after a `PUT`, and it scales to millions of rows. **Reminder:** a real export may contain **privileged or PII** data — that lands in a governed Snowflake account, never a teaching notebook.

### 📝 Recap — what you shipped

- Snowflake bulk-loads via **`PUT` → `stage` → `COPY INTO`**, controlled by a **file format** (`SKIP_HEADER`, delimiter) and **`ON_ERROR`**.
- A **stage** is Snowflake's file staging area; **`COPY INTO`** parses staged files into a table.
- You loaded a CSV into a table locally (the `COPY INTO` twin) and **reconciled** file vs. table row counts.
- **Bulk-load files; don't `INSERT` row-by-row** at scale.
- **Artifact:** `ingest(table, path)` — a reusable, self-verifying loader.

### 🧠 Check your understanding

1. Put the three Snowflake load steps in order: `COPY INTO`, `PUT`, choose a **file format**.
2. What does `SKIP_HEADER = 1` do, and what goes wrong if you forget it?
3. Why reconcile the file's row count against the table's after a load?
4. When should you bulk-load instead of `INSERT`?

<details><summary>✅ Answers</summary>

1. **`PUT`** the file to a stage → define/choose a **file format** → **`COPY INTO`** the table.
2. It skips the column-name row so it isn't loaded as data; forget it and you get one bogus "row" of header text (and a row count one too high).
3. To catch **silently dropped rows** — bad encodings, type mismatches, or wrong column counts don't always error; the count is your proof.
4. For anything **large** — `COPY INTO` (bulk) is far faster than many `INSERT`s; hand-`INSERT` only tiny/one-off data.
</details>

### ➡️ Next up — Week 7, Day 3: analytical SQL that shines in a warehouse

Now that data's *in*, we ask the questions a spreadsheet chokes on: **rank within groups**, **running totals**, **top-N per category** — with **window functions**, **`QUALIFY`**, and **CTEs**. This is where a warehouse earns its keep.

*Same toolkit, no install needed.*

### 📖 Reference & glossary

| Term | Plain meaning |
|---|---|
| **Stage** | Snowflake's holding area for files before loading |
| **`PUT`** | upload a local file into a stage |
| **File format** | rules for reading a file (CSV, header, delimiter) |
| **`COPY INTO`** | parse staged files and load rows into a table |
| **`SKIP_HEADER`** | ignore the column-name row |
| **`ON_ERROR`** | what to do with bad rows (`CONTINUE` / `ABORT_STATEMENT` / `SKIP_FILE`) |
| **Reconcile** | check loaded row count matches the source |
| **`read_csv_auto`** | DuckDB's local file loader (our `COPY INTO` twin) |

**Docs:** Snowflake `COPY INTO <table>` — https://docs.snowflake.com/en/sql-reference/sql/copy-into-table · Loading data overview — https://docs.snowflake.com/en/user-guide/data-load-overview

> *Not legal advice — these lessons teach technology. A lawyer reviews any AI or data output that will be relied upon.*